<a href="https://colab.research.google.com/github/Chiiyoo/UAS-Pengolahan-Citra/blob/dev/UAS_Deteksi_Uang_Koin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**INI PASTINYA FIX**

In [7]:
!pip install --upgrade gradio

import cv2
import numpy as np
import gradio as gr
import matplotlib
matplotlib.use('Agg') # WAJIB UNTUK COLAB agar grafik tidak crash
from matplotlib.figure import Figure
# import nest_asyncio
import math

# nest_asyncio.apply()

# =========================================================================
# HARDCODED REFERENCE & WEIGHTS (MURNI RULE-BASED)
# =========================================================================
REFERENSI_KOIN = {
    "Rp100": {
        "Diameter_mm" : 23.0,
        "Area_mm2"    : math.pi * (23.0 / 2) ** 2,
        "Perimeter_mm": math.pi * 23.0,
        "Circularity" : 0.90,
    },
    "Rp200": {
        "Diameter_mm" : 25.0,
        "Area_mm2"    : math.pi * (25.0 / 2) ** 2,
        "Perimeter_mm": math.pi * 25.0,
        "Circularity" : 0.90,
    },
    "Rp500": {
        "Diameter_mm" : 27.0,
        "Area_mm2"    : math.pi * (27.0 / 2) ** 2,
        "Perimeter_mm": math.pi * 27.0,
        "Circularity" : 0.90,
    },
    "Rp1000": {
        "Diameter_mm" : 24.15,
        "Area_mm2"    : math.pi * (24.15 / 2) ** 2,
        "Perimeter_mm": math.pi * 24.15,
        "Circularity" : 0.90,
    },
}

BOBOT_FITUR = {
    "Diameter_mm" : 0.60,
    "Area_mm2"    : 0.20,
    "Perimeter_mm": 0.05,
    "Circularity" : 0.15,
}

SIGMA_FITUR = {
    "Diameter_mm" : 1.0,
    "Area_mm2"    : 25.0,
    "Perimeter_mm": 3.2,
    "Circularity" : 0.05,
}

KONFIDENSI_MIN = 0.40

# =========================================================================
# FUNGSI BANTUAN (HELPER FUNCTIONS)
# =========================================================================
def hitung_hu_moments_log(contour):
    moments = cv2.moments(contour)
    hu = cv2.HuMoments(moments).flatten()
    for i in range(len(hu)):
        if hu[i] != 0:
            hu[i] = -1 * math.copysign(1.0, hu[i]) * math.log10(abs(hu[i]))
        else:
            hu[i] = 0
    return hu

def ekstrak_matriks_pusat(citra_gray, cx, cy):
    """Mengekstrak patch 5x5 secara dinamis berdasarkan koordinat centroid koin"""
    h, w = citra_gray.shape
    # Mengamankan agar koordinat tidak melewati batas array citra
    cy = max(2, min(h - 3, cy))
    cx = max(2, min(w - 3, cx))

    y1, y2 = cy - 2, cy + 3
    x1, x2 = cx - 2, cx + 3
    patch = citra_gray[y1:y2, x1:x2]
    return patch, int(citra_gray[cy, cx])

def generate_log_tahap(citra, tahap_num, nama_tahap, cx, cy):
    """Mencetak Log Matriks 5x5 tepat di tengah koin"""
    if len(citra.shape) == 3:
        citra_gray = cv2.cvtColor(citra, cv2.COLOR_BGR2GRAY)
    else:
        citra_gray = citra

    patch, center_val = ekstrak_matriks_pusat(citra_gray, cx, cy)
    mean_val = np.mean(patch)
    min_val = np.min(patch)
    max_val = np.max(patch)

    mat_str = ""
    for row in patch:
        mat_str += " ".join([f"{v:3}" for v in row]) + "\n"

    log_str = (
        f"======================\n"
        f"TAHAP {tahap_num}\n"
        f"{nama_tahap.upper()}\n"
        f"Target Center Pixel (X:{cx}, Y:{cy}) : {center_val}\n"
        f"Matrix 5x5\n"
        f"{mat_str}"
        f"Mean : {mean_val:.2f}\n"
        f"Min : {min_val}\n"
        f"Max : {max_val}\n"
        f"======================\n\n"
    )
    return log_str

# =========================================================================
# PIPELINE UTAMA
# =========================================================================
def pipeline_koin_stabil(gambar_input, val_bright, val_contrast, val_blur, val_t_block, val_t_c, val_ppm, val_auto_ppm):
    if gambar_input is None:
        blank_img = np.zeros((100,100), dtype=np.uint8)
        blank_fig = Figure(figsize=(5,3))
        blank_outputs = [(blank_img, blank_fig, "")] * 10
        flat_blank = [item for sublist in blank_outputs for item in sublist]
        return tuple(flat_blank + [[], "", blank_img, "⚠️ Unggah gambar terlebih dahulu."])

    if len(gambar_input.shape) == 3 and gambar_input.shape[2] == 4:
        img_raw = gambar_input[:, :, :3].copy()
    else:
        img_raw = gambar_input.copy()

    outputs = []

    # 1. SCALING
    lebar_target = 1000
    rasio = lebar_target / float(img_raw.shape[1])
    tinggi_target = int(img_raw.shape[0] * rasio)
    img_scaled = cv2.resize(img_raw, (lebar_target, tinggi_target), interpolation=cv2.INTER_AREA)

    # 2. SHADOW REMOVAL
    img_bgr = cv2.cvtColor(img_scaled, cv2.COLOR_RGB2BGR)
    kanals = cv2.split(img_bgr)
    kanals_bersih = []
    kernel_bg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    for kanal in kanals:
        bg_estimasi = cv2.morphologyEx(kanal, cv2.MORPH_CLOSE, kernel_bg)
        bg_estimasi = cv2.GaussianBlur(bg_estimasi, (21, 21), 0)
        kanal_compensated = cv2.divide(kanal, bg_estimasi, scale=255)
        kanals_bersih.append(kanal_compensated)
    img_shadow_removed = cv2.merge(kanals_bersih)
    img_shadow_rgb = cv2.cvtColor(img_shadow_removed, cv2.COLOR_BGR2RGB)

    # 3. GRAYSCALE
    gray = cv2.cvtColor(img_shadow_removed, cv2.COLOR_BGR2GRAY)

    # 4. BRIGHTNESS
    bright = cv2.convertScaleAbs(gray, alpha=1, beta=int(val_bright))

    # 5. CONTRAST
    faktor_c = max(0.1, float(val_contrast) / 50.0)
    contrast = np.clip((bright.astype(np.float32) - 128) * faktor_c + 128, 0, 255).astype(np.uint8)

    # 6. MEDIAN FILTER
    k_size = int(val_blur) if int(val_blur) % 2 != 0 else int(val_blur) + 1
    median_filtered = cv2.medianBlur(contrast, k_size)

    # 7. THRESHOLDING
    t_size = int(val_t_block) if int(val_t_block) % 2 != 0 else int(val_t_block) + 1
    thresh = cv2.adaptiveThreshold(median_filtered, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, t_size, int(val_t_c))

    # 8. MORPHOLOGY CLOSING
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    morph_closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close)
    temp_contours, _ = cv2.findContours(morph_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    morph_filled = np.zeros_like(morph_closed)
    cv2.drawContours(morph_filled, temp_contours, -1, 255, -1)

    # 9. MORPHOLOGY OPENING
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    morph_final = cv2.morphologyEx(morph_filled, cv2.MORPH_OPEN, kernel_open)

    # 10. CONTOUR DETECTION
    contours, _ = cv2.findContours(morph_final.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # ── MENCARI TITIK PUSAT KOIN PERTAMA (DYNAMIC TRACKING) ──
    sample_cx, sample_cy = img_scaled.shape[1] // 2, img_scaled.shape[0] // 2 # Fallback jika kosong
    for c in contours:
        if cv2.contourArea(c) > 3000:
            M = cv2.moments(c)
            if M["m00"] != 0:
                sample_cx = int(M["m10"] / M["m00"])
                sample_cy = int(M["m01"] / M["m00"])
                break

    img_contours = img_scaled.copy()
    cv2.drawContours(img_contours, contours, -1, (0, 255, 0), 2)
    # Gambar titik merah sebagai bukti lokasi pengambilan sampel Matriks 5x5
    cv2.circle(img_contours, (sample_cx, sample_cy), 8, (0, 0, 255), -1)

    # =========================================================================
    # VISUALISASI MATRIKS & HISTOGRAM
    # =========================================================================
    tahapan_pipeline = [
        (img_scaled, "Scaling"), (img_shadow_rgb, "Shadow Removal"),
        (gray, "Grayscale"), (bright, "Brightness"),
        (contrast, "Contrast"), (median_filtered, "Median Filter"),
        (thresh, "Thresholding"), (morph_filled, "Morphology Closing"),
        (morph_final, "Morphology Opening"), (img_contours, "Contour Detection")
    ]

    log_akademik = "=====================================================================\n"
    log_akademik += " 📄 LOG MATRIKS CITRA (DYNAMIC PIXEL TRACKING)\n"
    log_akademik += "=====================================================================\n\n"

    for i, (citra_tahap, nama_tahap) in enumerate(tahapan_pipeline, start=1):
        log_akademik += generate_log_tahap(citra_tahap, i, nama_tahap, sample_cx, sample_cy)

    fig = Figure(figsize=(12, 35))
    axes = fig.subplots(10, 2)

    for i, (citra_tahap, nama_tahap) in enumerate(tahapan_pipeline):
        # Kolom Kiri: Citra Visual
        ax_img = axes[i, 0]
        if len(citra_tahap.shape) == 3:
            citra_tampil = cv2.cvtColor(citra_tahap, cv2.COLOR_BGR2RGB) if citra_tahap is not img_scaled else citra_tahap
            ax_img.imshow(citra_tampil)
        else:
            ax_img.imshow(citra_tahap, cmap='gray')
        ax_img.set_title(f"Tahap {i+1}: {nama_tahap}")
        ax_img.axis('off')

        # Kolom Kanan: Histogram (LOGARITHMIC SCALE)
        ax_hist = axes[i, 1]
        c_gray = cv2.cvtColor(citra_tahap, cv2.COLOR_BGR2GRAY) if len(citra_tahap.shape) == 3 else citra_tahap
        # Menggunakan log=True agar sebaran piksel koin tidak tertutup oleh tingginya piksel background putih
        ax_hist.hist(c_gray.ravel(), 256, [0, 256], color='gray', alpha=0.7, log=True)
        ax_hist.set_title(f"Histogram: {nama_tahap} (Log Scale)")

    fig.tight_layout()

    # Isi ke output array
    for citra_tahap, _ in tahapan_pipeline:
        outputs.append(citra_tahap)
        outputs.append(None) # Placeholders for the individual plots in UI
        outputs.append(None) # Placeholders for individual texts in UI

    # Karena kita menggabungkan plot di akhir, kita perlu merekonstruksi output array
    outputs_fixed = []
    for citra_tahap, _ in tahapan_pipeline:
        outputs_fixed.extend([citra_tahap, None, None]) # Note: In Gradio, sending None to Plot won't update it.
        # But wait, to keep UI intact, let's just generate individual plots as required by the UI setup.

    # --- KOREKSI OUTPUT ARRAY AGAR SESUAI DENGAN UI ---
    outputs.clear()
    for i, (citra_tahap, nama_tahap) in enumerate(tahapan_pipeline, start=1):
        fig_ind = Figure(figsize=(5, 3))
        ax_ind = fig_ind.subplots()
        c_gray = cv2.cvtColor(citra_tahap, cv2.COLOR_BGR2GRAY) if len(citra_tahap.shape) == 3 else citra_tahap
        ax_ind.hist(c_gray.ravel(), 256, [0, 256], color='gray', alpha=0.7, log=True)
        ax_ind.set_title(f"{nama_tahap} (Log Scale)")
        fig_ind.tight_layout()

        outputs.append(citra_tahap)       # Image
        outputs.append(fig_ind)           # Plot
        outputs.append(generate_log_tahap(citra_tahap, i, nama_tahap, sample_cx, sample_cy)) # Text

    # ---------------------------------------------------------
    # TAHAP 11: FEATURE EXTRACTION
    # ---------------------------------------------------------
    valid_contours = []
    tabel_fitur = []

    for c in contours:
        hull = cv2.convexHull(c)
        area = cv2.contourArea(hull)
        if area < 3000:
            continue

        perimeter = cv2.arcLength(hull, True)
        circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

        if 0.70 <= circularity <= 1.25:
            ((cx, cy), radius) = cv2.minEnclosingCircle(hull)
            diameter_px  = radius * 2

            fitur = {
                "Diameter": diameter_px,
                "Radius": radius,
                "Area": area,
                "Perimeter": perimeter,
                "Circularity": circularity,
                "Contour_Raw": hull
            }
            valid_contours.append(fitur)

    if not valid_contours:
        outputs.append([])
        outputs.append("⚠️ Tidak ada objek koin yang terdeteksi.")
        outputs.append(img_scaled)
        outputs.append("Gagal: Threshold mungkin terlalu ketat.")
        return tuple(outputs)

    ppm = float(val_ppm) if float(val_ppm) > 0 else 1.0
    auto_calibrated = False

    if val_auto_ppm and len(valid_contours) >= 2:
        best_ppm = ppm
        min_total_error = float('inf')
        nominal_bi_diameters = [23.0, 25.0, 27.0, 24.15]

        for test_ppm in np.arange(4.0, 15.0, 0.02):
            total_error = 0.0
            for item in valid_contours:
                d_mm = item["Diameter"] / test_ppm
                closest_bi = min(nominal_bi_diameters, key=lambda x: abs(d_mm - x))
                total_error += (d_mm - closest_bi) ** 2

            if total_error < min_total_error:
                min_total_error = total_error
                best_ppm = test_ppm

        ppm = best_ppm
        auto_calibrated = True

    for i, item in enumerate(valid_contours):
        item["Diameter_mm"]  = item["Diameter"] / ppm
        item["Area_mm2"]     = item["Area"] / (ppm ** 2)
        item["Perimeter_mm"] = item["Perimeter"] / ppm

        tabel_fitur.append([
            f"Koin #{i+1}", f"{item['Area']:.1f}", f"{item['Perimeter']:.1f}",
            f"{item['Circularity']:.3f}", f"{item['Diameter']:.1f}",
            f"{item['Diameter_mm']:.2f}", f"{item['Area_mm2']:.2f}", f"{item['Perimeter_mm']:.2f}"
        ])

    outputs.append(tabel_fitur)

    # ---------------------------------------------------------
    # TAHAP 12: RULE BASED CLASSIFICATION
    # ---------------------------------------------------------
    log_klasifikasi  = "==========================================================\n"
    log_klasifikasi += "HASIL RULE BASED CLASSIFICATION (WEIGHTED SCORING - mm)\n"
    log_klasifikasi += "==========================================================\n\n"

    img_visualisasi = img_scaled.copy()
    koin_count = 0

    for item in valid_contours:
        koin_count += 1
        skor_nominal = {}

        for nominal, ref_val in REFERENSI_KOIN.items():
            skor_total = 0.0
            for fitur_nama, bobot in BOBOT_FITUR.items():
                val_ekstrak   = item[fitur_nama]
                val_referensi = ref_val[fitur_nama]
                sigma         = SIGMA_FITUR[fitur_nama]
                delta         = val_ekstrak - val_referensi
                similarity    = math.exp(-0.5 * (delta / sigma) ** 2)
                skor_total   += similarity * bobot
            skor_nominal[nominal] = skor_total

        hasil_prediksi = max(skor_nominal, key=skor_nominal.get)
        skor_terbaik   = skor_nominal[hasil_prediksi]

        label_tampil = "?" if skor_terbaik < KONFIDENSI_MIN else hasil_prediksi

        log_klasifikasi += f"KOIN #{koin_count}\n"
        log_klasifikasi += f"  Diameter Terdeteksi : {item['Diameter']:.1f} px  ->  {item['Diameter_mm']:.2f} mm\n"
        for nm in REFERENSI_KOIN:
            bintang = " <- TERPILIH" if nm == hasil_prediksi else ""
            log_klasifikasi += f"    {nm:<8}: {skor_nominal[nm]:.4f}{bintang}\n"
        log_klasifikasi += f"  Hasil Klasifikasi  => {label_tampil}\n"
        log_klasifikasi += f"----------------------------------------------------------\n"

        c_raw = item["Contour_Raw"]
        radius = item["Radius"]
        M = cv2.moments(c_raw)
        if M["m00"] != 0:
            cx_koin = int(M["m10"] / M["m00"])
            cy_koin = int(M["m01"] / M["m00"])
        else:
            ((cx_koin, cy_koin), _) = cv2.minEnclosingCircle(c_raw)
            cx_koin, cy_koin = int(cx_koin), int(cy_koin)

        cv2.circle(img_visualisasi, (cx_koin, cy_koin), int(radius), (0, 255, 0), 4)
        cv2.circle(img_visualisasi, (cx_koin, cy_koin), 5, (255, 0, 0), -1)
        cv2.putText(img_visualisasi, str(label_tampil),
                    (int(cx_koin - radius), int(cy_koin - radius - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 3)

    outputs.append(log_klasifikasi)
    outputs.append(img_visualisasi)
    outputs.append(f"🟢 Sukses memproses {koin_count} koin. PPM: {ppm:.3f} px/mm.")

    return tuple(outputs)

# ==============================================================================
# DESAIN ANTARMUKA GRADIO BLOCKS
# ==============================================================================
with gr.Blocks() as demo:
    gr.Markdown("# 🪙 SISTEM PIPELINE PENGOLAHAN CITRA DIGITAL (DETEKSI KOIN)")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Panel Media Input")
            ui_input_img = gr.Image(type="numpy", label="Foto Koin Utama")

            gr.Markdown("### 🎛️ Konfigurasi Variabel Spasial")
            ui_bright = gr.Slider(-100, 100, value=0, label="Konstanta Brightness")
            ui_contrast = gr.Slider(1, 200, value=50, label="Contrast Alpha")
            ui_blur = gr.Slider(3, 31, value=11, step=2, label="Mean Filter Kernel Size")
            ui_t_block = gr.Slider(3, 99, value=45, step=2, label="Adaptive Threshold Block Size")
            ui_t_c = gr.Slider(-20, 20, value=7, label="Adaptive Threshold Konstanta C")
            ui_ppm = gr.Slider(1.0, 35.0, value=7.00, step=0.05, label="📢 Kalibrasi PPM...")
            ui_auto_ppm = gr.Checkbox(label="💡 Auto-Kalibrasi PPM Dinamis", value=True)

            ui_btn = gr.Button("🚀 Analisis Seluruh Pipeline", variant="primary")

    gr.Markdown("---")

    ui_outputs = []

    tahap_names = [
        "Tahap 1 : Scaling", "Tahap 2 : Shadow Removal", "Tahap 3 : Grayscale",
        "Tahap 4 : Brightness", "Tahap 5 : Contrast", "Tahap 6 : Median Filter",
        "Tahap 7 : Thresholding", "Tahap 8 : Morphology Closing",
        "Tahap 9 : Morphology Opening", "Tahap 10: Contour Detection"
    ]

    for i, name in enumerate(tahap_names):
        gr.Markdown(f"### {name}")
        with gr.Row():
            img_out = gr.Image(label=f"Visualisasi {name}")
            hist_out = gr.Plot(label=f"Histogram {name}")
            log_out = gr.Textbox(label=f"Matriks Piksel {name}", lines=11)
            ui_outputs.extend([img_out, hist_out, log_out])

    gr.Markdown("---")

    gr.Markdown("### Tahap 11 : Feature Extraction")
    ui_table_fe = gr.Dataframe(
        headers=["ID", "Area(px²)", "Perimeter(px)", "Circularity",
                 "Diameter(px)", "Diameter(mm)", "Area(mm²)", "Perimeter(mm)"],
        datatype=["str", "str", "str", "str", "str", "str", "str", "str"],
        label="Tabel Ekstraksi Fitur (pixel & mm)"
    )
    ui_outputs.append(ui_table_fe)

    gr.Markdown("---")

    gr.Markdown("### Tahap 12 : Rule Based Classification")
    ui_log_rulebased = gr.Textbox(label="Log Klasifikasi Rule Based", lines=10)
    ui_outputs.append(ui_log_rulebased)

    gr.Markdown("---")

    gr.Markdown("### 🖼️ Hasil Akhir Identifikasi (Output Final)")
    with gr.Row():
        ui_out_img_final = gr.Image(label="Output Klasifikasi Nominal Koin")
        ui_status_final = gr.Textbox(label="Status Eksekusi", interactive=False)
    ui_outputs.extend([ui_out_img_final, ui_status_final])

    ui_btn.click(
        fn=pipeline_koin_stabil,
        inputs=[ui_input_img, ui_bright, ui_contrast, ui_blur, ui_t_block, ui_t_c, ui_ppm, ui_auto_ppm],
        outputs=ui_outputs
    )

demo.launch(debug=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fe3e62edb9435c00b7.gradio.live
